# MRI-to-PET Conditional Diffusion Model
## High-Fidelity Synthetic PET Generation
### Optimized for Kaggle T4 GPU (16GB VRAM)

A state-of-the-art conditional DDPM for synthesizing PET images from structural MRI scans.

**Key Features:**
- **2.5D Conditional Input**: 3 adjacent MRI slices for spatial context
- **Deep U-Net**: Residual blocks, GroupNorm, multi-head self-attention
- **Hybrid Loss**: Noise MSE + x0 L1 + differentiable SSIM + Laplacian pyramid + VGG perceptual
- **Classifier-Free Guidance (CFG)**: Amplified conditioning at inference for sharper outputs
- **Cosine Noise Schedule** with **DDIM Sampling** (50-step deterministic)
- **Self-Conditioning**: Feeds x0 prediction back as extra model input
- **EMA** (Exponential Moving Average) for stable generation
- **Comprehensive Augmentation**: Flip, rotation, intensity, noise, elastic, gamma, bias field
- **Mixed Precision**: FP16 for fast T4 training
- **Hyperparameter Tuning**: See `Hyperparameter_Tuning.ipynb`

**Data:** NACC preprocessed MRI-PET pairs (160 x 192 x 160, normalized to [-1, 1])

**Target:** SSIM > 0.90 on validation set

In [ ]:
# ============================================================
# SETUP: Install Dependencies + Import Libraries
# ============================================================

import os, sys, gc, math, copy, random, time, json
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as compare_ssim
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from scipy.ndimage import rotate as scipy_rotate
from scipy.ndimage import map_coordinates, gaussian_filter
import torchvision.models as tv_models
from tqdm.notebook import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================================
# CONFIGURATION — All hyperparameters in one place
# ============================================================
# To tune hyperparameters, run Hyperparameter_Tuning.ipynb and
# copy the best values back here.

class Config:
    # ---- Paths (CHANGE THESE) ----
    # Kaggle:  "/kaggle/input/your-dataset-name"
    # Local:   r"D:\NACC_Matched_Pairs_500_Preprocessed"
    DATA_ROOT = "/kaggle/input/nacc-preprocessed"
    SAVE_DIR  = "/kaggle/working/diffusion_checkpoints"

    # ---- Data ----
    IMG_H, IMG_W    = 160, 192
    MIN_BRAIN_PX    = 800
    PET_NONBG_THRESH = -0.9
    MIN_PET_PX      = 500
    MIN_PET_STD     = 0.03
    TRAIN_SPLIT     = 0.70
    VAL_SPLIT       = 0.15
    TEST_SPLIT      = 0.15

    # ---- Model ----
    BASE_CH         = 64
    CH_MULT         = (1, 2, 4, 8)    # -> 64, 128, 256, 512
    NUM_RES_BLOCKS  = 2
    ATTN_LEVELS     = (2, 3)
    TIME_EMB_DIM    = 256
    DROPOUT         = 0.05

    # ---- Diffusion ----
    NUM_TIMESTEPS   = 500
    SCHEDULE        = "cosine"

    # ---- Training ----
    BATCH_SIZE      = 6
    EPOCHS          = 200
    LR              = 5e-5
    WEIGHT_DECAY    = 1e-4
    EMA_DECAY       = 0.995
    GRAD_CLIP       = 1.0
    WARMUP_EPOCHS   = 10

    # ---- Hybrid Loss Weights ----
    W_NOISE         = 1.0    # Combined global+brain-weighted noise objective
    W_RECON         = 0.5    # L1 on predicted x0, SNR-weighted
    W_SSIM          = 0.2    # Differentiable SSIM loss on x0
    W_PYRAMID       = 0.2    # Laplacian pyramid loss on x0
    W_PERCEPTUAL    = 0.1    # VGG perceptual loss on x0
    W_BG_ANCHOR     = 0.05   # Push non-brain background back toward -1

    # ---- Classifier-Free Guidance ----
    CFG_DROP_PROB   = 0.1    # P(drop condition) during training
    CFG_SCALE       = 1.3    # Start lower; high CFG can amplify noise artifacts

    # ---- Self-Conditioning ----
    SELF_COND_PROB  = 0.5    # P(self-conditioning) during training

    # ---- Patch Training ----
    PATCH_SIZE      = 128    # Random crop size for training (None = full slice)
    SMART_CROP      = True   # Prefer crops around PET foreground when available

    # ---- Augmentation ----
    AUG_FLIP_P      = 0.5
    AUG_ROT_DEG     = 5.0
    AUG_ROT_P       = 0.3
    AUG_INTENSITY   = 0.05
    AUG_INTENSITY_P = 0.3
    AUG_NOISE_STD   = 0.02
    AUG_NOISE_P     = 0.2
    AUG_ELASTIC_P       = 0.3
    AUG_ELASTIC_ALPHA   = 80.0
    AUG_ELASTIC_SIGMA   = 10.0
    AUG_GAMMA_P         = 0.3
    AUG_GAMMA_RANGE     = (0.8, 1.2)
    AUG_BIAS_FIELD_P    = 0.2

    # ---- Sampling / Eval ----
    DDIM_STEPS      = 50
    DDIM_ETA        = 0.0
    VAL_EVERY       = 5
    NUM_VAL_SAMPLES = 100
    SAMPLE_EVERY    = 5      # Generate visual samples every N epochs

    # ---- Resume (auto-detected, but can override) ----
    RESUME_PATH     = ""     # e.g. "/kaggle/input/your-checkpoints/latest_checkpoint.pt"
    SAVE_EVERY      = 2      # Save resume checkpoint every N epochs

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
for attr in sorted(vars(Config)):
    if not attr.startswith('_'):
        print(f"  {attr:20s} = {getattr(cfg, attr)}")
print("=" * 60)

## 1. Data Pipeline

**Preprocessing** (already done by `fixing_size.py`):
MRI and PET registered, percentile-normalized to [-1, 1], resampled to (160, 192, 160).

**2.5D Slicing**: For target PET slice `k`, MRI slices `[k-1, k, k+1]` form 3-channel input.

**Augmentation** (critical for 200 patients):
- Random horizontal flip (p=0.5)
- Random rotation ±5° (p=0.3) — applied to both MRI and PET
- Random intensity scaling ±5% (p=0.3) — MRI only
- Random Gaussian noise σ=0.02 (p=0.2) — MRI only (PET stays clean as target)

**Classifier-Free Guidance prep**: 10% of samples have MRI condition zeroed out.

In [ ]:
# ============================================================
# DATASET WITH AUGMENTATION + CLASSIFIER-FREE GUIDANCE
# ============================================================

def gamma_transform(image, gamma):
    """Apply gamma intensity transform. Image expected in [-1, 1]."""
    img_01 = np.clip((image + 1.0) / 2.0, 0, 1)
    img_01 = np.power(img_01, gamma)
    return (img_01 * 2.0 - 1.0).astype(np.float32)


def random_bias_field(shape, order=3):
    """Generate a smooth multiplicative bias field using low-order polynomial.
    Returns a field with values centered around 1.0."""
    H, W = shape
    y = np.linspace(-1, 1, H)
    x = np.linspace(-1, 1, W)
    yy, xx = np.meshgrid(y, x, indexing='ij')
    coeffs = np.random.normal(0, 0.03, size=(order + 1, order + 1))
    coeffs[0, 0] = 1.0
    field = np.zeros((H, W), dtype=np.float32)
    for i in range(order + 1):
        for j in range(order + 1):
            if i + j <= order:
                field += coeffs[i, j] * (yy ** i) * (xx ** j)
    return field


def augment_pair(mri_ctx, pet_sl, cfg):
    """Apply matched augmentations to MRI-PET pair.
    Geometric augs applied to both; appearance augs to MRI only."""
    # Horizontal flip (both)
    if random.random() < cfg.AUG_FLIP_P:
        mri_ctx = np.flip(mri_ctx, axis=2).copy()
        pet_sl  = np.flip(pet_sl, axis=2).copy()

    # Rotation (both)
    if cfg.AUG_ROT_DEG > 0 and random.random() < cfg.AUG_ROT_P:
        angle = random.uniform(-cfg.AUG_ROT_DEG, cfg.AUG_ROT_DEG)
        for c in range(mri_ctx.shape[0]):
            mri_ctx[c] = scipy_rotate(mri_ctx[c], angle, reshape=False,
                                       order=1, mode='nearest')
        pet_sl[0] = scipy_rotate(pet_sl[0], angle, reshape=False,
                                  order=1, mode='nearest')

    # Elastic deformation (both MRI and PET — geometric)
    if random.random() < cfg.AUG_ELASTIC_P:
        shape_2d = mri_ctx.shape[1:]  # (H, W)
        dx = gaussian_filter(np.random.randn(*shape_2d), cfg.AUG_ELASTIC_SIGMA) * cfg.AUG_ELASTIC_ALPHA
        dy = gaussian_filter(np.random.randn(*shape_2d), cfg.AUG_ELASTIC_SIGMA) * cfg.AUG_ELASTIC_ALPHA
        y, x = np.meshgrid(np.arange(shape_2d[0]), np.arange(shape_2d[1]), indexing='ij')
        indices = [np.clip(y + dy, 0, shape_2d[0] - 1),
                   np.clip(x + dx, 0, shape_2d[1] - 1)]
        for c in range(mri_ctx.shape[0]):
            mri_ctx[c] = map_coordinates(mri_ctx[c], indices, order=1, mode='reflect').astype(np.float32)
        pet_sl[0] = map_coordinates(pet_sl[0], indices, order=1, mode='reflect').astype(np.float32)

    # Intensity scaling (MRI only)
    if cfg.AUG_INTENSITY > 0 and random.random() < cfg.AUG_INTENSITY_P:
        s = 1.0 + random.uniform(-cfg.AUG_INTENSITY, cfg.AUG_INTENSITY)
        mri_ctx = np.clip(mri_ctx * s, -1.0, 1.0).astype(np.float32)

    # Gaussian noise (MRI only)
    if cfg.AUG_NOISE_STD > 0 and random.random() < cfg.AUG_NOISE_P:
        noise = np.random.normal(0, cfg.AUG_NOISE_STD, mri_ctx.shape).astype(np.float32)
        mri_ctx = np.clip(mri_ctx + noise, -1.0, 1.0)

    # Gamma transform (MRI only — intensity)
    if random.random() < cfg.AUG_GAMMA_P:
        gamma = random.uniform(cfg.AUG_GAMMA_RANGE[0], cfg.AUG_GAMMA_RANGE[1])
        for c in range(mri_ctx.shape[0]):
            mri_ctx[c] = gamma_transform(mri_ctx[c], gamma)

    # Random bias field (MRI only — intensity)
    if random.random() < cfg.AUG_BIAS_FIELD_P:
        bias = random_bias_field(mri_ctx.shape[1:])
        for c in range(mri_ctx.shape[0]):
            mri_ctx[c] = np.clip(mri_ctx[c] * bias, -1.0, 1.0).astype(np.float32)

    return mri_ctx, pet_sl


class MRIPETSliceDataset(Dataset):
    """Loads MRI-PET volume pairs and yields 2D axial slices with 2.5D context."""

    def __init__(self, patient_dirs, cfg, augment=True):
        self.cfg = cfg
        self.augment = augment
        self.patch_size = cfg.PATCH_SIZE if augment else None
        self.patients = []
        self.samples  = []

        for pdir in tqdm(patient_dirs, desc="Indexing"):
            mri_p = os.path.join(pdir, "mri_processed.npy")
            pet_p = os.path.join(pdir, "pet_processed.npy")
            if not (os.path.exists(mri_p) and os.path.exists(pet_p)):
                continue
            try:
                mri = np.load(mri_p, mmap_mode='r')
                pet = np.load(pet_p, mmap_mode='r')
                if mri.shape != (cfg.IMG_H, cfg.IMG_W, 160):
                    continue
                if pet.shape != mri.shape:
                    continue

                pidx = len(self.patients)
                self.patients.append((mri_p, pet_p))
                D = mri.shape[2]

                for k in range(1, D - 1):
                    mri_k = np.array(mri[:, :, k])
                    pet_k = np.array(pet[:, :, k])
                    mri_fg = np.sum(mri_k > cfg.PET_NONBG_THRESH)
                    pet_fg = np.sum(pet_k > cfg.PET_NONBG_THRESH)
                    pet_vals = pet_k[pet_k > cfg.PET_NONBG_THRESH]
                    pet_std = float(np.std(pet_vals)) if pet_vals.size > 10 else 0.0

                    if mri_fg > cfg.MIN_BRAIN_PX and pet_fg > cfg.MIN_PET_PX and pet_std >= cfg.MIN_PET_STD:
                        self.samples.append((pidx, k))
            except Exception:
                continue

        print(f"  -> {len(self.patients)} patients, {len(self.samples)} valid slices")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pidx, k = self.samples[idx]
        mri_p, pet_p = self.patients[pidx]
        mri = np.load(mri_p, mmap_mode='r')
        pet = np.load(pet_p, mmap_mode='r')

        # 2.5D context: 3 adjacent MRI slices
        mri_ctx = np.stack([
            np.array(mri[:, :, k - 1], dtype=np.float32),
            np.array(mri[:, :, k],     dtype=np.float32),
            np.array(mri[:, :, k + 1], dtype=np.float32),
        ], axis=0)  # (3, H, W)
        pet_sl = np.array(pet[:, :, k], dtype=np.float32)[None]  # (1, H, W)

        # Data augmentation
        if self.augment:
            mri_ctx, pet_sl = augment_pair(mri_ctx.copy(), pet_sl.copy(), self.cfg)

        # Classifier-free guidance: randomly drop condition
        if self.augment and random.random() < self.cfg.CFG_DROP_PROB:
            mri_ctx = np.zeros_like(mri_ctx)

        # Patch-based crop (training only)
        if self.patch_size is not None and self.patch_size < mri_ctx.shape[1]:
            H, W = mri_ctx.shape[1], mri_ctx.shape[2]
            ps = self.patch_size

            top, left = None, None
            if self.cfg.SMART_CROP:
                fg = pet_sl[0] > self.cfg.PET_NONBG_THRESH
                ys, xs = np.where(fg)
                if len(ys) > 50:
                    cy = int(np.median(ys))
                    cx = int(np.median(xs))
                    jitter = ps // 6
                    cy += random.randint(-jitter, jitter)
                    cx += random.randint(-jitter, jitter)
                    top = np.clip(cy - ps // 2, 0, H - ps)
                    left = np.clip(cx - ps // 2, 0, W - ps)

            if top is None or left is None:
                top = random.randint(0, H - ps)
                left = random.randint(0, W - ps)

            mri_ctx = mri_ctx[:, top:top+ps, left:left+ps].copy()
            pet_sl  = pet_sl[:, top:top+ps, left:left+ps].copy()

        return torch.from_numpy(mri_ctx.copy()), torch.from_numpy(pet_sl.copy())


# ---- Build datasets (70% train / 15% val / 15% test) ----
print("Building datasets...")
root = Path(cfg.DATA_ROOT)
all_dirs = sorted([str(d) for d in root.iterdir() if d.is_dir()])
print(f"Found {len(all_dirs)} patient directories")

random.seed(SEED)
random.shuffle(all_dirs)
n_test = max(1, int(len(all_dirs) * cfg.TEST_SPLIT))
n_val  = max(1, int(len(all_dirs) * cfg.VAL_SPLIT))
test_dirs  = all_dirs[:n_test]
val_dirs   = all_dirs[n_test:n_test + n_val]
train_dirs = all_dirs[n_test + n_val:]
print(f"Split: {len(train_dirs)} train / {len(val_dirs)} val / {len(test_dirs)} test patients")

train_dataset = MRIPETSliceDataset(train_dirs, cfg, augment=True)
val_dataset   = MRIPETSliceDataset(val_dirs,   cfg, augment=False)
test_dataset  = MRIPETSliceDataset(test_dirs,  cfg, augment=False)

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)

print(f"\nTrain: {len(train_dataset):,} slices ({len(train_loader):,} batches/epoch)")
print(f"Val:   {len(val_dataset):,} slices ({len(val_loader):,} batches)")
print(f"Test:  {len(test_dataset):,} slices ({len(test_loader):,} batches)")

# Sanity check
mri_s, pet_s = train_dataset[0]
pet_fg = (pet_s[0].numpy() > cfg.PET_NONBG_THRESH)
print(f"\nSample shapes — MRI: {mri_s.shape}, PET: {pet_s.shape}")
print(f"MRI range: [{mri_s.min():.2f}, {mri_s.max():.2f}]")
print(f"PET range: [{pet_s.min():.2f}, {pet_s.max():.2f}]")
print(f"PET foreground pixels in sample: {pet_fg.sum()}")

In [ ]:
# ============================================================
# VISUALIZE SAMPLE TRAINING PAIRS
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    idx = random.randint(0, len(train_dataset) - 1)
    mri_s, pet_s = train_dataset[idx]
    axes[0, i].imshow(mri_s[1].numpy(), cmap='gray', vmin=-1, vmax=1)
    axes[0, i].set_title(f'MRI (slice {idx})')
    axes[0, i].axis('off')
    axes[1, i].imshow(pet_s[0].numpy(), cmap='hot', vmin=-1, vmax=1)
    axes[1, i].set_title('PET (target)')
    axes[1, i].axis('off')
plt.suptitle('Sample Training Pairs (MRI top, PET bottom)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(cfg.SAVE_DIR, 'sample_pairs.png'), dpi=100)
plt.show()

avg_slices = len(train_dataset) / max(1, len(train_dataset.patients))
print(f"Average valid slices per patient: {avg_slices:.0f}")
print(f"Total training samples (with augmentation ~2x): ~{len(train_dataset) * 2:,}")

In [ ]:
# ============================================================
# QUICK PET TARGET SANITY CHECK (distribution + dynamic range)
# ============================================================
# If PET is almost constant near -1, the model can only learn noisy residuals.

def inspect_pet_distribution(dataset, n_samples=128):
    vals = []
    fg_fracs = []
    n = min(n_samples, len(dataset))
    for _ in range(n):
        idx = random.randint(0, len(dataset) - 1)
        _, pet_s = dataset[idx]
        p = pet_s.numpy()[0]
        vals.append(p.reshape(-1))
        fg_fracs.append((p > cfg.PET_NONBG_THRESH).mean())

    vals = np.concatenate(vals)
    p2, p50, p98 = np.percentile(vals, [2, 50, 98])
    print(f"PET intensity percentiles: p2={p2:.3f}, p50={p50:.3f}, p98={p98:.3f}")
    print(f"Mean PET foreground fraction (> {cfg.PET_NONBG_THRESH}): {np.mean(fg_fracs):.3f}")

    if p98 - p2 < 0.15:
        print("WARNING: PET dynamic range is very low; check preprocessing normalization.")

    plt.figure(figsize=(7, 4))
    plt.hist(vals, bins=100, alpha=0.8, edgecolor='black')
    plt.title('PET target intensity histogram (train samples)')
    plt.xlabel('Intensity'); plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

inspect_pet_distribution(train_dataset, n_samples=128)

In [ ]:
# ============================================================
# DATA VERIFICATION: Check MRI-PET Alignment Quality
# ============================================================
# Verifies that MRI and PET volumes are properly aligned/registered.
# Checks: spatial overlap, brain mask correlation, per-patient quality.
# Flags bad patients so they can be excluded.

print("=" * 70)
print("DATA VERIFICATION: Checking MRI-PET alignment quality")
print("=" * 70)

def verify_patient_alignment(mri_path, pet_path, patient_id, slices_to_check=5):
    """Check alignment between MRI and PET for one patient.
    Returns dict with quality metrics, or None if files can't be loaded."""
    try:
        mri = np.load(mri_path)
        pet = np.load(pet_path)
    except Exception as e:
        return {'patient': patient_id, 'error': str(e)}

    if mri.shape != pet.shape:
        return {'patient': patient_id, 'error': f'Shape mismatch: MRI {mri.shape} vs PET {pet.shape}'}

    H, W, D = mri.shape

    # Brain masks (voxels with actual content, not background)
    mri_brain = mri > -0.9
    pet_brain = pet > -0.9

    # 1. Volume overlap: do MRI and PET have brain in similar regions?
    mri_brain_frac = mri_brain.sum() / mri_brain.size
    pet_brain_frac = pet_brain.sum() / pet_brain.size
    overlap = (mri_brain & pet_brain).sum()
    union   = (mri_brain | pet_brain).sum()
    dice    = 2.0 * overlap / (mri_brain.sum() + pet_brain.sum() + 1e-8)
    iou     = overlap / (union + 1e-8)

    # 2. Slice-level correlation in brain region across mid slices
    mid = D // 2
    check_slices = list(range(max(1, mid - slices_to_check), min(D - 1, mid + slices_to_check + 1)))
    correlations = []
    ssim_vals    = []
    for k in check_slices:
        mri_sl = mri[:, :, k].astype(np.float64)
        pet_sl = pet[:, :, k].astype(np.float64)
        mask = (mri_sl > -0.9) & (pet_sl > -0.9)
        if mask.sum() < 200:
            continue
        # Pearson correlation in brain region
        m_vals = mri_sl[mask]
        p_vals = pet_sl[mask]
        if m_vals.std() > 1e-6 and p_vals.std() > 1e-6:
            corr = np.corrcoef(m_vals, p_vals)[0, 1]
            correlations.append(corr)
        # SSIM on the full slice
        ssim_val = compare_ssim(mri_sl.astype(np.float64), pet_sl.astype(np.float64), data_range=2.0)
        ssim_vals.append(ssim_val)

    # 3. Check if PET has center of mass near MRI center of mass
    mri_coords = np.array(np.where(mri_brain))
    pet_coords = np.array(np.where(pet_brain))
    if mri_coords.shape[1] > 0 and pet_coords.shape[1] > 0:
        mri_com = mri_coords.mean(axis=1)
        pet_com = pet_coords.mean(axis=1)
        com_dist = np.linalg.norm(mri_com - pet_com)
    else:
        com_dist = 999.0

    # 4. PET signal quality: check if PET has reasonable dynamic range
    pet_nonzero = pet[pet > -0.9]
    pet_range = pet_nonzero.max() - pet_nonzero.min() if len(pet_nonzero) > 100 else 0.0

    return {
        'patient': patient_id,
        'mri_brain_frac': float(mri_brain_frac),
        'pet_brain_frac': float(pet_brain_frac),
        'dice': float(dice),
        'iou': float(iou),
        'mean_corr': float(np.mean(correlations)) if correlations else 0.0,
        'mean_ssim': float(np.mean(ssim_vals)) if ssim_vals else 0.0,
        'com_distance': float(com_dist),
        'pet_range': float(pet_range),
    }


# Run verification on ALL patients
all_patient_dirs = sorted([str(d) for d in Path(cfg.DATA_ROOT).iterdir() if d.is_dir()])
verification_results = []
bad_patients = []

for pdir in tqdm(all_patient_dirs, desc="Verifying alignment"):
    patient_id = os.path.basename(pdir)
    mri_p = os.path.join(pdir, "mri_processed.npy")
    pet_p = os.path.join(pdir, "pet_processed.npy")
    if not (os.path.exists(mri_p) and os.path.exists(pet_p)):
        continue
    result = verify_patient_alignment(mri_p, pet_p, patient_id)
    verification_results.append(result)

    # Flag bad patients
    if 'error' in result:
        bad_patients.append((patient_id, result['error']))
    elif result['dice'] < 0.3:
        bad_patients.append((patient_id, f"Low Dice overlap: {result['dice']:.3f}"))
    elif result['com_distance'] > 30.0:
        bad_patients.append((patient_id, f"Large center-of-mass shift: {result['com_distance']:.1f} voxels"))
    elif result['pet_range'] < 0.3:
        bad_patients.append((patient_id, f"Low PET dynamic range: {result['pet_range']:.3f}"))

# Aggregate stats
valid = [r for r in verification_results if 'error' not in r]
if valid:
    print(f"\n{'='*70}")
    print(f"VERIFICATION SUMMARY ({len(valid)} patients)")
    print(f"{'='*70}")
    dice_vals = [r['dice'] for r in valid]
    corr_vals = [r['mean_corr'] for r in valid]
    ssim_vals = [r['mean_ssim'] for r in valid]
    com_vals  = [r['com_distance'] for r in valid]
    pet_range = [r['pet_range'] for r in valid]

    print(f"  Brain Dice overlap:  {np.mean(dice_vals):.3f} +/- {np.std(dice_vals):.3f}  (min: {np.min(dice_vals):.3f})")
    print(f"  MRI-PET correlation: {np.mean(corr_vals):.3f} +/- {np.std(corr_vals):.3f}  (min: {np.min(corr_vals):.3f})")
    print(f"  MRI-PET SSIM:        {np.mean(ssim_vals):.3f} +/- {np.std(ssim_vals):.3f}  (min: {np.min(ssim_vals):.3f})")
    print(f"  Center-of-mass dist: {np.mean(com_vals):.1f}  +/- {np.std(com_vals):.1f}   (max: {np.max(com_vals):.1f} voxels)")
    print(f"  PET dynamic range:   {np.mean(pet_range):.3f} +/- {np.std(pet_range):.3f}  (min: {np.min(pet_range):.3f})")

if bad_patients:
    print(f"\n  WARNING: {len(bad_patients)} potentially problematic patients:")
    for pid, reason in bad_patients[:20]:
        print(f"    - {pid}: {reason}")
    if len(bad_patients) > 20:
        print(f"    ... and {len(bad_patients) - 20} more")
else:
    print(f"\n  All {len(valid)} patients passed alignment checks.")

# Visualize alignment for 6 patients (3 best + 3 worst by Dice)
if len(valid) >= 6:
    sorted_by_dice = sorted(valid, key=lambda r: r['dice'])
    show_patients = sorted_by_dice[:3] + sorted_by_dice[-3:]
    labels = ['WORST'] * 3 + ['BEST'] * 3

    fig, axes = plt.subplots(len(show_patients), 4, figsize=(18, 3.5 * len(show_patients)))
    for row, (result, label) in enumerate(zip(show_patients, labels)):
        pdir = os.path.join(cfg.DATA_ROOT, result['patient'])
        mri = np.load(os.path.join(pdir, "mri_processed.npy"))
        pet = np.load(os.path.join(pdir, "pet_processed.npy"))
        mid_k = mri.shape[2] // 2

        mri_sl = mri[:, :, mid_k]
        pet_sl = pet[:, :, mid_k]

        # MRI
        axes[row, 0].imshow(mri_sl, cmap='gray', vmin=-1, vmax=1)
        axes[row, 0].set_title(f'{label} - MRI ({result["patient"][:15]})')
        axes[row, 0].axis('off')

        # PET
        axes[row, 1].imshow(pet_sl, cmap='hot', vmin=-1, vmax=1)
        axes[row, 1].set_title(f'PET (Dice={result["dice"]:.3f})')
        axes[row, 1].axis('off')

        # Overlay: MRI in gray, PET contours in red
        axes[row, 2].imshow(mri_sl, cmap='gray', vmin=-1, vmax=1)
        pet_mask = pet_sl > -0.5
        axes[row, 2].contour(pet_mask, levels=[0.5], colors='red', linewidths=1)
        axes[row, 2].set_title(f'Overlay (corr={result["mean_corr"]:.3f})')
        axes[row, 2].axis('off')

        # Side-by-side brain masks
        mri_mask = mri_sl > -0.9
        pet_mask2 = pet_sl > -0.9
        combined = np.zeros((*mri_sl.shape, 3))
        combined[:, :, 0] = pet_mask2.astype(float)  # PET in red
        combined[:, :, 1] = mri_mask.astype(float)    # MRI in green
        combined[:, :, 2] = (mri_mask & pet_mask2).astype(float)  # overlap in blue
        axes[row, 3].imshow(combined)
        axes[row, 3].set_title(f'Masks (R=PET, G=MRI, overlap=yellow)')
        axes[row, 3].axis('off')

    plt.suptitle('Data Verification: MRI-PET Alignment (3 worst + 3 best by Dice)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.SAVE_DIR, 'data_verification.png'), dpi=150)
    plt.show()

# Histogram of Dice scores
if valid:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist([r['dice'] for r in valid], bins=20, edgecolor='black', alpha=0.7)
    axes[0].axvline(x=0.3, color='r', ls='--', label='Threshold (0.3)')
    axes[0].set_xlabel('Dice Score'); axes[0].set_ylabel('Count')
    axes[0].set_title('Brain Mask Dice Overlap'); axes[0].legend()

    axes[1].hist([r['mean_corr'] for r in valid], bins=20, edgecolor='black', alpha=0.7, color='orange')
    axes[1].set_xlabel('Pearson Correlation'); axes[1].set_ylabel('Count')
    axes[1].set_title('MRI-PET Correlation (brain region)')

    axes[2].hist([r['com_distance'] for r in valid], bins=20, edgecolor='black', alpha=0.7, color='green')
    axes[2].axvline(x=30, color='r', ls='--', label='Threshold (30)')
    axes[2].set_xlabel('Distance (voxels)'); axes[2].set_ylabel('Count')
    axes[2].set_title('Center-of-Mass Distance'); axes[2].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.SAVE_DIR, 'data_verification_histograms.png'), dpi=150)
    plt.show()

print(f"\nVerification complete. {len(valid)} patients checked, {len(bad_patients)} flagged.")

## 2. Model Architecture

### Conditional U-Net with Classifier-Free Guidance Support

| Component | Detail |
|---|---|
| **Input** | `cat(noisy_PET[1ch], MRI_context[3ch])` = 4 channels |
| **Output** | Predicted noise ε (1 channel) |
| **Encoder** | 160x192/64 → 80x96/128 → 40x48/256+Attn → 20x24/512+Attn |
| **Bottleneck** | ResBlock → Attention → ResBlock (512ch @ 20x24) |
| **Decoder** | 20x24/512 → 40x48/256 → 80x96/128 → 160x192/64 (with skip connections) |

- **Residual Blocks**: GroupNorm → SiLU → Conv → time embedding injection → GroupNorm → SiLU → Dropout → Conv + skip
- **Self-Attention**: Multi-head attention at low resolutions (levels 2,3) with Flash Attention support
- **Sinusoidal Time Embedding**: 256-dim encoding projected to 1024-dim via MLP

In [ ]:
# ============================================================
# MODEL BUILDING BLOCKS
# ============================================================

def _num_groups(channels, max_groups=32):
    """Find largest valid num_groups <= max_groups that evenly divides channels."""
    for g in range(min(max_groups, channels), 0, -1):
        if channels % g == 0:
            return g
    return 1


class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device, dtype=torch.float32) * -emb)
        emb = t.float()[:, None] * emb[None, :]
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)


class ResBlock(nn.Module):
    """GroupNorm -> SiLU -> Conv -> (+time) -> GroupNorm -> SiLU -> Drop -> Conv + skip."""
    def __init__(self, in_ch, out_ch, time_dim, dropout=0.0):
        super().__init__()
        self.norm1  = nn.GroupNorm(_num_groups(in_ch), in_ch)
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(time_dim, out_ch)
        self.norm2  = nn.GroupNorm(_num_groups(out_ch), out_ch)
        self.drop   = nn.Dropout(dropout)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = F.silu(self.norm1(x))
        h = self.conv1(h)
        h = h + self.t_proj(F.silu(t_emb))[:, :, None, None]
        h = F.silu(self.norm2(h))
        h = self.drop(h)
        h = self.conv2(h)
        return h + self.skip(x)


class SelfAttention(nn.Module):
    """Multi-head self-attention with residual. Uses Flash Attention when available."""
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = channels // num_heads
        self.norm = nn.GroupNorm(_num_groups(channels), channels)
        self.qkv  = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, self.num_heads, self.head_dim, H * W)
        qkv = qkv.permute(1, 0, 2, 4, 3)  # (3, B, heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if hasattr(F, 'scaled_dot_product_attention'):
            out = F.scaled_dot_product_attention(q, k, v)
        else:
            scale = self.head_dim ** -0.5
            attn = (q @ k.transpose(-2, -1)) * scale
            attn = F.softmax(attn, dim=-1)
            out = attn @ v
        out = out.permute(0, 1, 3, 2).reshape(B, C, H, W)
        return x + self.proj(out)


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))

print("Building blocks defined.")

In [ ]:
# ============================================================
# CONDITIONAL U-NET (with Self-Conditioning support)
# ============================================================

class ConditionalUNet(nn.Module):
    """Conditional U-Net for MRI->PET diffusion with CFG and self-conditioning.
    Input:  cat(noisy_PET[1ch], MRI_context[3ch], x0_prev[1ch]) = 5 channels
    Output: Predicted noise [1ch]
    """
    def __init__(self, in_channels=5, out_channels=1, base_ch=64,
                 ch_mult=(1, 2, 4, 8), num_res_blocks=2,
                 attn_levels=(2, 3), time_emb_dim=256, dropout=0.1):
        super().__init__()
        channels  = [base_ch * m for m in ch_mult]
        n_levels  = len(channels)
        t_dim     = time_emb_dim * 4

        # Time embedding MLP
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, t_dim), nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )

        # Input projection
        self.conv_in = nn.Conv2d(in_channels, channels[0], 3, padding=1)

        # Encoder
        self.encoder    = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        prev_ch = channels[0]
        for lv in range(n_levels):
            ch = channels[lv]
            blocks = nn.ModuleList()
            blocks.append(ResBlock(prev_ch, ch, t_dim, dropout))
            for _ in range(num_res_blocks - 1):
                blocks.append(ResBlock(ch, ch, t_dim, dropout))
            if lv in attn_levels:
                blocks.append(SelfAttention(ch, max(1, ch // 64)))
            self.encoder.append(blocks)
            prev_ch = ch
            if lv < n_levels - 1:
                self.downsamples.append(Downsample(ch))

        # Bottleneck
        mid = channels[-1]
        self.mid_block1 = ResBlock(mid, mid, t_dim, dropout)
        self.mid_attn   = SelfAttention(mid, max(1, mid // 64))
        self.mid_block2 = ResBlock(mid, mid, t_dim, dropout)

        # Decoder
        self.decoder   = nn.ModuleList()
        self.upsamples = nn.ModuleList()
        prev_ch = mid
        for lv in reversed(range(n_levels)):
            ch = channels[lv]
            blocks = nn.ModuleList()
            blocks.append(ResBlock(prev_ch + ch, ch, t_dim, dropout))
            for _ in range(num_res_blocks - 1):
                blocks.append(ResBlock(ch, ch, t_dim, dropout))
            if lv in attn_levels:
                blocks.append(SelfAttention(ch, max(1, ch // 64)))
            self.decoder.append(blocks)
            if lv > 0:
                self.upsamples.append(Upsample(ch))
            prev_ch = ch

        # Output
        self.out_norm = nn.GroupNorm(_num_groups(channels[0]), channels[0])
        self.out_conv = nn.Conv2d(channels[0], out_channels, 3, padding=1)

    def forward(self, x, t, cond, x0_prev=None):
        """x: (B,1,H,W) noisy PET, t: (B,) timesteps, cond: (B,3,H,W) MRI,
           x0_prev: (B,1,H,W) previous x0 prediction for self-conditioning (or None/zeros)."""
        if x0_prev is None:
            x0_prev = torch.zeros_like(x)
        h = self.conv_in(torch.cat([x, cond, x0_prev], dim=1))  # 1+3+1 = 5 channels
        t_emb = self.time_embed(t)

        skips = []
        for lv, blocks in enumerate(self.encoder):
            for block in blocks:
                h = block(h, t_emb) if isinstance(block, ResBlock) else block(h)
            skips.append(h)
            if lv < len(self.downsamples):
                h = self.downsamples[lv](h)

        h = self.mid_block1(h, t_emb)
        h = self.mid_attn(h)
        h = self.mid_block2(h, t_emb)

        up_idx = 0
        for blocks in self.decoder:
            h = torch.cat([h, skips.pop()], dim=1)
            for block in blocks:
                h = block(h, t_emb) if isinstance(block, ResBlock) else block(h)
            if up_idx < len(self.upsamples):
                h = self.upsamples[up_idx](h)
                up_idx += 1

        return self.out_conv(F.silu(self.out_norm(h)))


def build_model(cfg):
    """Factory function to build model from config."""
    return ConditionalUNet(
        in_channels=5, out_channels=1,
        base_ch=cfg.BASE_CH, ch_mult=cfg.CH_MULT,
        num_res_blocks=cfg.NUM_RES_BLOCKS,
        attn_levels=tuple(cfg.ATTN_LEVELS),
        time_emb_dim=cfg.TIME_EMB_DIM, dropout=cfg.DROPOUT,
    )


def count_params(model):
    return sum(p.numel() for p in model.parameters())

# Quick size check
_m = build_model(cfg)
n_params = count_params(_m)
param_mb = n_params * 4 / 1e6
print(f"Model parameters: {n_params:,}")
print(f"Model size: {param_mb:.0f} MB (FP32)")
print(f"Estimated VRAM: ~{param_mb * 4:.0f} MB (model+optimizer+grads+EMA)")
del _m; gc.collect()

## 3. Diffusion Process & Hybrid Loss

### Cosine Noise Schedule (Nichol & Dhariwal 2021)
1000 timesteps with cosine beta schedule for better SNR distribution.

### Hybrid Loss (key to high SSIM)
Instead of just noise MSE, we compute the implied x₀ from the predicted noise and add:
1. **Noise MSE** (w=1.0): Standard ε-prediction loss
2. **Reconstruction L1** (w=0.3): |x₀_pred - x₀|, weighted by α̅ₜ (active at low noise)
3. **SSIM Loss** (w=0.2): Differentiable SSIM on x₀_pred, SNR-weighted
4. **Laplacian Pyramid** (w=0.1): Multi-scale L1 for structural preservation

The SNR weighting (α̅ₜ) ensures reconstruction losses only contribute meaningfully at low noise levels.

### DDIM Sampling with Classifier-Free Guidance
- `ε = ε_uncond + scale × (ε_cond - ε_uncond)` with scale=2.5
- 50-step deterministic sampling (η=0)

In [ ]:
# ============================================================
# DIFFUSION PROCESS + LOSS FUNCTIONS
# ============================================================

def cosine_beta_schedule(T, s=0.008):
    steps = T + 1
    x = torch.linspace(0, T, steps)
    ac = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
    ac = ac / ac[0]
    betas = 1 - (ac[1:] / ac[:-1])
    return torch.clamp(betas, 0.0001, 0.9999)


class GaussianDiffusion:
    """Forward diffusion (q_sample) and reverse sampling (DDIM with CFG + self-conditioning)."""

    def __init__(self, T=500, schedule='cosine'):
        self.T = T
        if schedule == 'cosine':
            betas = cosine_beta_schedule(T)
        else:
            betas = torch.linspace(1e-4, 0.02, T)

        self.betas = betas
        alphas = 1.0 - betas
        self.alphas_cumprod = torch.cumprod(alphas, dim=0)
        self.sqrt_ac        = torch.sqrt(self.alphas_cumprod)
        self.sqrt_1m_ac     = torch.sqrt(1.0 - self.alphas_cumprod)

        # For posterior
        self.ac_prev     = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.post_var    = betas * (1.0 - self.ac_prev) / (1.0 - self.alphas_cumprod)

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sa = self.sqrt_ac.to(x0.device)[t].view(-1, 1, 1, 1)
        sm = self.sqrt_1m_ac.to(x0.device)[t].view(-1, 1, 1, 1)
        return sa * x0 + sm * noise

    def predict_x0(self, x_t, t, eps_pred):
        """Recover x0 from x_t and predicted noise."""
        sa = self.sqrt_ac.to(x_t.device)[t].view(-1, 1, 1, 1)
        sm = self.sqrt_1m_ac.to(x_t.device)[t].view(-1, 1, 1, 1)
        return ((x_t - sm * eps_pred) / sa.clamp(min=1e-8)).clamp(-1, 1)

    @torch.no_grad()
    def ddim_sample(self, model, cond, shape, steps=50, eta=0.0, cfg_scale=1.0):
        """DDIM sampling with CFG and self-conditioning."""
        device = cond.device
        B = shape[0]
        times = torch.linspace(self.T - 1, 0, steps + 1).long()
        x = torch.randn(shape, device=device)
        x0_prev = torch.zeros(shape, device=device)  # self-conditioning init

        for i in tqdm(range(len(times) - 1), desc="DDIM", leave=False):
            t_now  = times[i].item()
            t_next = times[i + 1].item()
            t_b = torch.full((B,), t_now, device=device, dtype=torch.long)

            # Classifier-free guidance with self-conditioning
            if cfg_scale > 1.0:
                x_double = torch.cat([x, x], dim=0)
                t_double = torch.cat([t_b, t_b], dim=0)
                c_double = torch.cat([cond, torch.zeros_like(cond)], dim=0)
                x0_prev_double = torch.cat([x0_prev, x0_prev], dim=0)
                with autocast('cuda', enabled=(device.type == 'cuda')):
                    eps_double = model(x_double, t_double, c_double, x0_prev_double)
                eps_cond, eps_uncond = eps_double.chunk(2, dim=0)
                eps = eps_uncond + cfg_scale * (eps_cond - eps_uncond)
            else:
                with autocast('cuda', enabled=(device.type == 'cuda')):
                    eps = model(x, t_b, cond, x0_prev)

            a_t    = self.alphas_cumprod.to(device)[t_now]
            a_next = self.alphas_cumprod.to(device)[max(t_next, 0)]

            x0_pred = ((x - torch.sqrt(1 - a_t) * eps) / torch.sqrt(a_t)).clamp(-1, 1)
            x0_prev = x0_pred  # feed to next step as self-conditioning

            if t_next <= 0:
                x = x0_pred
            else:
                sigma = eta * torch.sqrt((1 - a_next) / (1 - a_t) * (1 - a_t / a_next))
                dir_xt = torch.sqrt(1 - a_next - sigma ** 2) * eps
                x = torch.sqrt(a_next) * x0_pred + dir_xt
                if eta > 0:
                    x = x + sigma * torch.randn_like(x)
        return x


# ---- Differentiable SSIM Loss ----
def _gaussian_window(size, sigma, channels, device):
    coords = torch.arange(size, dtype=torch.float32, device=device) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window = g.unsqueeze(1) @ g.unsqueeze(0)
    return window.unsqueeze(0).unsqueeze(0).repeat(channels, 1, 1, 1)

def ssim_loss(pred, target, window_size=11, data_range=2.0):
    """Differentiable SSIM loss. Returns 1 - SSIM (minimize to maximize SSIM)."""
    C = pred.shape[1]
    window = _gaussian_window(window_size, 1.5, C, pred.device)
    pad = window_size // 2

    mu_p = F.conv2d(pred, window, padding=pad, groups=C)
    mu_t = F.conv2d(target, window, padding=pad, groups=C)
    mu_pp, mu_tt, mu_pt = mu_p ** 2, mu_t ** 2, mu_p * mu_t

    sig_pp = F.conv2d(pred * pred, window, padding=pad, groups=C) - mu_pp
    sig_tt = F.conv2d(target * target, window, padding=pad, groups=C) - mu_tt
    sig_pt = F.conv2d(pred * target, window, padding=pad, groups=C) - mu_pt

    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2

    ssim_map = ((2 * mu_pt + C1) * (2 * sig_pt + C2)) / \
               ((mu_pp + mu_tt + C1) * (sig_pp + sig_tt + C2))
    return 1.0 - ssim_map.mean()


# ---- Laplacian Pyramid Loss ----
def laplacian_pyramid_loss(pred, target, n_levels=4):
    """Multi-scale L1 loss for structural fidelity."""
    total = 0.0
    weights = [2 ** (n_levels - 1 - i) for i in range(n_levels)]
    w_sum = sum(weights)
    for i in range(n_levels):
        total += F.l1_loss(pred, target) * weights[i]
        if i < n_levels - 1:
            pred   = F.avg_pool2d(pred, 2)
            target = F.avg_pool2d(target, 2)
    return total / w_sum


# ---- Perceptual Loss (VGG16 features) ----
class VGGPerceptualLoss(nn.Module):
    """Perceptual loss using VGG16 features[:16] (up to relu3_3).
    Single-channel PET is repeated to 3 channels for VGG input."""

    def __init__(self):
        super().__init__()
        vgg = tv_models.vgg16(weights=tv_models.VGG16_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(vgg.features[:16]))
        for p in self.features.parameters():
            p.requires_grad_(False)
        self.features.eval()
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        """pred, target: (B, 1, H, W) in [-1, 1]."""
        pred_01   = (pred + 1.0) / 2.0
        target_01 = (target + 1.0) / 2.0
        pred_3ch   = pred_01.repeat(1, 3, 1, 1)
        target_3ch = target_01.repeat(1, 3, 1, 1)
        pred_3ch   = (pred_3ch - self.mean) / self.std
        target_3ch = (target_3ch - self.mean) / self.std
        with torch.no_grad():
            target_feat = self.features(target_3ch)
        pred_feat = self.features(pred_3ch)
        return F.l1_loss(pred_feat, target_feat)


# Instantiate VGG perceptual loss
vgg_loss_fn = VGGPerceptualLoss().to(device)


# ---- Combined Hybrid Loss (with background anchoring) ----
def compute_hybrid_loss(eps_pred, eps_true, x_t, x_0, t, diffusion, cfg):
    """Compute hybrid loss: noise MSE + SNR-weighted (L1 + SSIM + Laplacian + Perceptual) on x0.
    Adds background anchoring to reduce persistent speckle in non-brain regions."""

    # Soft region weighting
    brain_mask = (x_0 > cfg.PET_NONBG_THRESH).float()
    bg_mask = 1.0 - brain_mask
    brain_sum = brain_mask.sum().clamp(min=1.0)
    bg_sum = bg_mask.sum().clamp(min=1.0)

    # 1. Noise loss: combine global stability + brain emphasis
    noise_global = F.mse_loss(eps_pred, eps_true)
    noise_brain = (((eps_pred - eps_true) ** 2) * brain_mask).sum() / brain_sum
    noise_loss = 0.5 * noise_global + 0.5 * noise_brain

    # Predict x0 for reconstruction losses
    x0_pred = diffusion.predict_x0(x_t, t, eps_pred)

    # SNR weight: high at low noise (small t), low at high noise (large t)
    ac_t = diffusion.alphas_cumprod.to(x_t.device)[t].view(-1, 1, 1, 1)

    # 2. Reconstruction L1 (full + brain emphasis)
    l1_full = F.l1_loss(x0_pred, x_0)
    l1_brain = (F.l1_loss(x0_pred, x_0, reduction='none') * brain_mask).sum() / brain_sum
    recon_loss = (0.5 * l1_full + 0.5 * l1_brain) * ac_t.mean()

    # 3. SSIM loss (SNR-weighted)
    ssim_w = ac_t.mean()
    s_loss = ssim_loss(x0_pred, x_0) * ssim_w

    # 4. Laplacian pyramid loss (SNR-weighted)
    pyr_loss = laplacian_pyramid_loss(x0_pred, x_0) * ssim_w

    # 5. Perceptual loss (SNR-weighted)
    perc_loss = vgg_loss_fn(x0_pred, x_0) * ssim_w

    # 6. Background anchor: enforce clean dark background around -1
    bg_anchor = (((x0_pred + 1.0) ** 2) * bg_mask).sum() / bg_sum

    total = (cfg.W_NOISE       * noise_loss +
             cfg.W_RECON       * recon_loss +
             cfg.W_SSIM        * s_loss +
             cfg.W_PYRAMID     * pyr_loss +
             cfg.W_PERCEPTUAL  * perc_loss +
             cfg.W_BG_ANCHOR   * bg_anchor)

    return total, {
        'noise': noise_loss.item(),
        'noise_global': noise_global.item(),
        'recon': recon_loss.item(),
        'ssim':  s_loss.item(),
        'pyramid': pyr_loss.item(),
        'perceptual': perc_loss.item(),
        'bg_anchor': bg_anchor.item(),
        'total': total.item(),
    }


diffusion = GaussianDiffusion(cfg.NUM_TIMESTEPS, cfg.SCHEDULE)
print(f"Diffusion: {cfg.NUM_TIMESTEPS} timesteps, {cfg.SCHEDULE} schedule")
print(f"Loss: noise({cfg.W_NOISE}) + recon({cfg.W_RECON}) + ssim({cfg.W_SSIM}) "
      f"+ pyramid({cfg.W_PYRAMID}) + perceptual({cfg.W_PERCEPTUAL}) + bg({cfg.W_BG_ANCHOR})")

## 4. Training

- **Optimizer**: AdamW (lr=2e-4, weight_decay=1e-4)
- **LR Schedule**: Cosine annealing with 5-epoch linear warmup
- **EMA**: Exponential Moving Average (decay=0.995) for stable inference
- **Mixed Precision**: FP16 (GradScaler) for T4 speed
- **Gradient Clipping**: max_norm=1.0
- **Checkpointing**: Save best model (by SSIM) + resume checkpoint every 2 epochs
- **Validation**: Every 5 epochs, DDIM-sample 100 slices and compute SSIM/PSNR/MAE

### Resume Support
Training saves **complete state** (model, EMA, optimizer, scheduler, scaler) every `SAVE_EVERY` epochs.
If your Kaggle session disconnects:
1. Download `latest_checkpoint.pt` from the output
2. Upload it as a Kaggle dataset (or to your input folder)
3. Set `RESUME_PATH` in Config to point to the checkpoint
4. Re-run the notebook — training resumes exactly where it stopped

In [ ]:
# ============================================================
# TRAINING UTILITIES: EMA, train_one_epoch, evaluate, samples
# ============================================================

class EMA:
    """Exponential Moving Average of model parameters."""
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.model = copy.deepcopy(model)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, p in zip(self.model.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(p.data, alpha=1 - self.decay)


def get_lr_lambda(epoch, warmup_epochs, total_epochs):
    """Linear warmup + cosine decay."""
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def train_one_epoch(model, loader, optimizer, diffusion, cfg, scaler=None):
    """Train one epoch with hybrid loss, mixed precision, and self-conditioning."""
    model.train()
    loss_accum = {}
    n_batches = 0

    for mri_cond, pet_target in tqdm(loader, desc="Train", leave=False):
        mri_cond   = mri_cond.to(device, non_blocking=True)
        pet_target = pet_target.to(device, non_blocking=True)
        B = pet_target.shape[0]
        t = torch.randint(0, diffusion.T, (B,), device=device)
        noise = torch.randn_like(pet_target)
        x_t = diffusion.q_sample(pet_target, t, noise)

        # Self-conditioning: with probability SELF_COND_PROB, compute x0_prev
        x0_prev = torch.zeros_like(pet_target)
        if random.random() < cfg.SELF_COND_PROB:
            with torch.no_grad():
                if scaler is not None:
                    with autocast('cuda'):
                        eps_prev = model(x_t, t, mri_cond, x0_prev)
                else:
                    eps_prev = model(x_t, t, mri_cond, x0_prev)
                x0_prev = diffusion.predict_x0(x_t, t, eps_prev).detach()

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with autocast('cuda'):
                eps_pred = model(x_t, t, mri_cond, x0_prev)
                loss, breakdown = compute_hybrid_loss(
                    eps_pred, noise, x_t, pet_target, t, diffusion, cfg)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            eps_pred = model(x_t, t, mri_cond, x0_prev)
            loss, breakdown = compute_hybrid_loss(
                eps_pred, noise, x_t, pet_target, t, diffusion, cfg)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

        for k, v in breakdown.items():
            if k not in loss_accum:
                loss_accum[k] = 0.0
            loss_accum[k] += v
        n_batches += 1

    return {k: v / max(1, n_batches) for k, v in loss_accum.items()}


@torch.no_grad()
def evaluate_model(model, loader, diffusion, cfg, max_samples=100):
    """Evaluate using DDIM+CFG sampling. Returns SSIM, brain-SSIM, PSNR, MAE."""
    model.eval()
    ssim_all, brain_ssim_all, psnr_all, mae_all = [], [], [], []
    count = 0

    for mri_cond, pet_target in tqdm(loader, desc="Eval", leave=False):
        if count >= max_samples:
            break
        mri_cond = mri_cond.to(device, non_blocking=True)
        syn_pet = diffusion.ddim_sample(
            model, mri_cond, pet_target.shape,
            steps=cfg.DDIM_STEPS, eta=cfg.DDIM_ETA, cfg_scale=cfg.CFG_SCALE)

        syn_np  = syn_pet.cpu().numpy()
        real_np = pet_target.numpy()

        for b in range(syn_np.shape[0]):
            if count >= max_samples:
                break
            s = syn_np[b, 0]
            r = real_np[b, 0]

            ssim_val = compare_ssim(r, s, data_range=2.0)
            ssim_all.append(ssim_val)

            _, ssim_map = compare_ssim(r, s, data_range=2.0, full=True)
            mask = r > -0.9
            if mask.sum() > 100:
                brain_ssim_all.append(ssim_map[mask].mean())

            psnr_all.append(compare_psnr(r, s, data_range=2.0))
            mae_all.append(np.mean(np.abs(r - s)))
            count += 1

    return {
        'ssim':       np.mean(ssim_all) if ssim_all else 0.0,
        'ssim_std':   np.std(ssim_all)  if ssim_all else 0.0,
        'brain_ssim': np.mean(brain_ssim_all) if brain_ssim_all else 0.0,
        'psnr':       np.mean(psnr_all) if psnr_all else 0.0,
        'mae':        np.mean(mae_all)  if mae_all  else 0.0,
        'n_samples':  count,
    }


@torch.no_grad()
def generate_sample_images(model, loader, diffusion, cfg, epoch, n_samples=4):
    """Generate side-by-side MRI | Real PET | Synthetic PET visualizations.

    Produces a grid saved to SAVE_DIR/samples_epoch_XXX.png so you can
    visually track how the generated PET quality improves over training.
    """
    model.eval()
    mri_batch, pet_batch = next(iter(loader))
    mri_batch = mri_batch[:n_samples].to(device, non_blocking=True)
    pet_batch = pet_batch[:n_samples]

    syn_pet = diffusion.ddim_sample(
        model, mri_batch, (n_samples, 1, pet_batch.shape[2], pet_batch.shape[3]),
        steps=cfg.DDIM_STEPS, eta=cfg.DDIM_ETA, cfg_scale=cfg.CFG_SCALE)
    syn_np  = syn_pet.cpu().numpy()
    real_np = pet_batch.numpy()
    mri_np  = mri_batch.cpu().numpy()

    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
    if n_samples == 1:
        axes = axes[None, :]
    for i in range(n_samples):
        # MRI (center slice of 2.5D context)
        axes[i, 0].imshow(mri_np[i, 1], cmap='gray', vmin=-1, vmax=1)
        axes[i, 0].set_title('MRI Input' if i == 0 else '')
        axes[i, 0].axis('off')
        # Real PET
        axes[i, 1].imshow(real_np[i, 0], cmap='hot', vmin=-1, vmax=1)
        axes[i, 1].set_title('Real PET' if i == 0 else '')
        axes[i, 1].axis('off')
        # Synthetic PET
        ssim_val = compare_ssim(real_np[i, 0], syn_np[i, 0], data_range=2.0)
        axes[i, 2].imshow(syn_np[i, 0], cmap='hot', vmin=-1, vmax=1)
        axes[i, 2].set_title(f'Synthetic PET (SSIM={ssim_val:.3f})' if i == 0
                             else f'SSIM={ssim_val:.3f}')
        axes[i, 2].axis('off')

    fig.suptitle(f'Epoch {epoch}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    save_path = os.path.join(cfg.SAVE_DIR, f'samples_epoch_{epoch:03d}.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  >> Sample images saved: {save_path}")


print("Training utilities defined.")

In [ ]:
# ============================================================
# MAIN TRAINING LOOP (with full resume support)
# ============================================================

def run_training(cfg, train_loader, val_loader, diffusion, resume_path=None):
    """Full training loop with robust checkpointing for resume after interruption.

    Saves ALL training state (model, EMA, optimizer, scheduler, scaler, epoch,
    metrics history) so training can resume exactly where it left off.
    Checkpoints are saved every cfg.SAVE_EVERY epochs (default: 2).
    Sample images are generated every cfg.SAMPLE_EVERY epochs (default: 5).
    """

    model = build_model(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR,
                                  weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda ep: get_lr_lambda(ep, cfg.WARMUP_EPOCHS, cfg.EPOCHS))
    ema = EMA(model, decay=cfg.EMA_DECAY)
    scaler = GradScaler('cuda') if device.type == 'cuda' else None

    best_ssim = 0.0
    start_epoch = 1
    train_losses = []
    val_history  = []

    # ---- Resume from checkpoint ----
    if resume_path and os.path.exists(resume_path):
        print(f"Loading checkpoint: {resume_path}")
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        ema.model.load_state_dict(ckpt['ema_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt.get('epoch', 0) + 1
        best_ssim   = ckpt.get('best_ssim', 0.0)
        train_losses = ckpt.get('train_losses', [])
        val_history  = ckpt.get('val_history', [])

        # Restore scheduler state
        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        else:
            for _ in range(start_epoch - 1):
                scheduler.step()

        if scaler is not None and 'scaler_state_dict' in ckpt:
            scaler.load_state_dict(ckpt['scaler_state_dict'])

        print(f"  Resumed from epoch {start_epoch - 1}")
        print(f"  Best SSIM so far: {best_ssim:.4f}")
        print(f"  Training losses recorded: {len(train_losses)} epochs")
        print(f"  Remaining: epochs {start_epoch} -> {cfg.EPOCHS}")

    if start_epoch > cfg.EPOCHS:
        print(f"Training already complete (epoch {start_epoch - 1} >= {cfg.EPOCHS}).")
        return model, ema, train_losses, val_history, best_ssim

    n_params = count_params(model)
    print(f"Model: {n_params:,} params | Training epochs {start_epoch}-{cfg.EPOCHS}")
    print(f"Checkpoint frequency: every {cfg.SAVE_EVERY} epochs")
    print(f"Sample generation: every {cfg.SAMPLE_EVERY} epochs")
    print("=" * 70)

    for epoch in range(start_epoch, cfg.EPOCHS + 1):
        t0 = time.time()
        losses = train_one_epoch(model, train_loader, optimizer, diffusion, cfg, scaler)
        scheduler.step()
        ema.update(model)
        train_losses.append(losses['total'])

        lr_now = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0
        print(f"Epoch {epoch:3d}/{cfg.EPOCHS} | Loss: {losses['total']:.5f} "
              f"(n:{losses['noise']:.4f} r:{losses['recon']:.4f} "
              f"s:{losses['ssim']:.4f} p:{losses['pyramid']:.4f} "
              f"v:{losses.get('perceptual', 0):.4f}) | "
              f"LR: {lr_now:.2e} | {elapsed:.0f}s")

        # Validation
        if epoch % cfg.VAL_EVERY == 0 or epoch == cfg.EPOCHS:
            metrics = evaluate_model(ema.model, val_loader, diffusion, cfg,
                                     cfg.NUM_VAL_SAMPLES)
            val_history.append((epoch, metrics))
            print(f"  >> Val SSIM: {metrics['ssim']:.4f}+/-{metrics['ssim_std']:.3f} | "
                  f"Brain: {metrics['brain_ssim']:.4f} | "
                  f"PSNR: {metrics['psnr']:.1f}dB | MAE: {metrics['mae']:.4f}")

            if metrics['ssim'] > best_ssim:
                best_ssim = metrics['ssim']
                _save_checkpoint(cfg, model, ema, optimizer, scheduler, scaler,
                                 epoch, best_ssim, train_losses, val_history,
                                 metrics=metrics, filename='best_model.pt')
                print(f"  >> NEW BEST (SSIM={best_ssim:.4f})")

        # Generate sample images to visualize progress
        if epoch % cfg.SAMPLE_EVERY == 0 or epoch == cfg.EPOCHS:
            generate_sample_images(ema.model, val_loader, diffusion, cfg, epoch)

        # Periodic resume checkpoint (every SAVE_EVERY epochs)
        if epoch % cfg.SAVE_EVERY == 0 or epoch == cfg.EPOCHS:
            _save_checkpoint(cfg, model, ema, optimizer, scheduler, scaler,
                             epoch, best_ssim, train_losses, val_history,
                             filename='latest_checkpoint.pt')
            print(f"  >> Checkpoint saved (epoch {epoch})")

    print("=" * 70)
    print(f"Training complete. Best SSIM: {best_ssim:.4f}")

    return model, ema, train_losses, val_history, best_ssim


def _save_checkpoint(cfg, model, ema, optimizer, scheduler, scaler,
                     epoch, best_ssim, train_losses, val_history,
                     metrics=None, filename='latest_checkpoint.pt'):
    """Save a complete checkpoint with all state needed for resume."""
    ckpt = {
        'epoch': epoch,
        'best_ssim': best_ssim,
        'model_state_dict': model.state_dict(),
        'ema_state_dict': ema.model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_losses': train_losses,
        'val_history': val_history,
        'cfg': {k: v for k, v in vars(cfg).items() if not k.startswith('_')},
    }
    if scaler is not None:
        ckpt['scaler_state_dict'] = scaler.state_dict()
    if metrics is not None:
        ckpt['metrics'] = metrics
    torch.save(ckpt, os.path.join(cfg.SAVE_DIR, filename))


# ---- Run training ----
print("Starting training...")
resume = cfg.RESUME_PATH if cfg.RESUME_PATH else os.path.join(cfg.SAVE_DIR, 'latest_checkpoint.pt')
if not os.path.exists(resume):
    resume = None

model, ema, train_losses, val_history, best_ssim = run_training(
    cfg, train_loader, val_loader, diffusion, resume_path=resume)

In [ ]:
# ============================================================
# LOAD BEST MODEL + FINAL COMPREHENSIVE EVALUATION
# ============================================================

ckpt_path = os.path.join(cfg.SAVE_DIR, 'best_model.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = build_model(cfg).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    ema = EMA(model, decay=cfg.EMA_DECAY)
    ema.model.load_state_dict(ckpt['ema_state_dict'])
    eval_model = ema.model
    eval_model.eval()
    print(f"Loaded best model from epoch {ckpt['epoch']}")
    print(f"Checkpoint SSIM: {ckpt['metrics']['ssim']:.4f}")
    train_losses = ckpt.get('train_losses', train_losses)
    val_history  = ckpt.get('val_history', val_history)
else:
    eval_model = ema.model
    eval_model.eval()
    print("Using current EMA model (no best checkpoint found).")

# ---- Training curves ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, linewidth=0.8)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Total Loss')
axes[0].set_title('Training Loss'); axes[0].grid(True, alpha=0.3)

if val_history:
    ep_v = [v[0] for v in val_history]
    ss   = [v[1]['ssim'] for v in val_history]
    bs   = [v[1]['brain_ssim'] for v in val_history]
    axes[1].plot(ep_v, ss, 'o-', label='Full SSIM')
    axes[1].plot(ep_v, bs, 's--', label='Brain SSIM')
    axes[1].axhline(y=0.90, color='r', ls=':', alpha=0.5, label='Target 0.90')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('SSIM')
    axes[1].set_title('Validation SSIM'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()

# ---- Full validation evaluation ----
print("\nRunning final evaluation on FULL validation set...")
final_val = evaluate_model(eval_model, val_loader, diffusion, cfg,
                           max_samples=len(val_dataset))
print(f"\n{'='*50}")
print(f"VALIDATION METRICS ({final_val['n_samples']} samples)")
print(f"{'='*50}")
print(f"  SSIM:        {final_val['ssim']:.4f} +/- {final_val['ssim_std']:.4f}")
print(f"  Brain SSIM:  {final_val['brain_ssim']:.4f}")
print(f"  PSNR:        {final_val['psnr']:.2f} dB")
print(f"  MAE:         {final_val['mae']:.4f}")
print(f"{'='*50}")

# ---- Full test evaluation ----
print("\nRunning final evaluation on FULL test set...")
final_test = evaluate_model(eval_model, test_loader, diffusion, cfg,
                            max_samples=len(test_dataset))
print(f"\n{'='*50}")
print(f"TEST METRICS ({final_test['n_samples']} samples)")
print(f"{'='*50}")
print(f"  SSIM:        {final_test['ssim']:.4f} +/- {final_test['ssim_std']:.4f}")
print(f"  Brain SSIM:  {final_test['brain_ssim']:.4f}")
print(f"  PSNR:        {final_test['psnr']:.2f} dB")
print(f"  MAE:         {final_test['mae']:.4f}")
print(f"{'='*50}")

# ---- Generate final sample visualizations on test set ----
generate_sample_images(eval_model, test_loader, diffusion, cfg, epoch=cfg.EPOCHS, n_samples=4)

In [ ]:
## Summary

### Architecture & Training

| Component | Detail |
|---|---|
| **Model** | Conditional U-Net (64/128/256/512 channels) with self-attention + self-conditioning |
| **Input** | 2.5D MRI context (3 adjacent axial slices) + self-conditioning channel |
| **Diffusion** | 500 timesteps, cosine noise schedule |
| **Sampling** | DDIM (50 steps, deterministic) + Classifier-Free Guidance (scale=2.0) |
| **Training** | AdamW, warmup+cosine LR, EMA (0.995), mixed precision FP16 |
| **Loss** | Hybrid: noise MSE + SNR-weighted (L1 + SSIM + Laplacian pyramid + VGG perceptual), brain-mask weighted |
| **Augmentation** | Flip, rotation, intensity scaling, Gaussian noise, elastic deformation, gamma transform, bias field |
| **HP Tuning** | See `Hyperparameter_Tuning.ipynb` (separate notebook) |
| **Resume** | Full state checkpoint every 2 epochs (model+EMA+optimizer+scheduler+scaler) |

### Key Design Decisions for High SSIM
1. **Hybrid loss** directly optimizes SSIM alongside standard noise prediction
2. **VGG perceptual loss** preserves structural features beyond pixel-level
3. **Brain-mask weighting** focuses learning on brain tissue, ignoring background
4. **Self-conditioning** feeds x0 prediction back as extra model input
5. **Classifier-free guidance** amplifies the MRI conditioning signal at inference
6. **Patch training** (128x128 crops) for memory-efficient training
7. **SNR-weighted reconstruction losses** focus x0 quality at low-noise timesteps
8. **Strong augmentation** (elastic, gamma, bias field) prevents overfitting on 200 patients
9. **EMA model** provides smoother, more consistent generations
10. **Cosine schedule** gives better SNR distribution than linear

### Outputs
- `best_model.pt` — best checkpoint (by validation SSIM)
- `latest_checkpoint.pt` — resume-capable checkpoint (every 2 epochs)
- `data_verification.png` — MRI-PET alignment visualization
- `data_verification_histograms.png` — distribution of alignment metrics
- `training_curves.png` — loss and SSIM plots
- `synthesis_results.png` — side-by-side MRI/PET comparison
- `3d_volume_comparison.png` — full 3D volume multi-view comparison
- `sample_synthetic_pet.npy` — example generated 3D PET volume

### Usage on Kaggle

**First run (with HP tuning):**
1. Upload preprocessed data as a Kaggle dataset
2. Run `Hyperparameter_Tuning.ipynb` to find best hyperparameters
3. Copy the best values into the `Config` class in this notebook
4. Run this notebook for full training

**If session disconnects:**
1. Download `latest_checkpoint.pt` from the output
2. Upload it as a new Kaggle dataset (e.g., `nacc-checkpoints`)
3. Set `RESUME_PATH = "/kaggle/input/nacc-checkpoints/latest_checkpoint.pt"`
4. Re-run the notebook — training resumes from checkpoint

**Subsequent runs:**
- `latest_checkpoint.pt` is auto-detected — training continues from last saved epoch
- No manual config changes needed if files are in `SAVE_DIR`